In [13]:
import pandas as pd
import json
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

df = pd.read_csv("../data/processed/training_data_with_history.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

with open("../data/processed/selected_features.json", "r") as file:
    raw_features = json.load(file)

history_features = ["smart_5_raw_change_1d", "smart_5_raw_change_7d", "smart_5_raw_change_30d", "smart_5_raw_rolling_mean_7d", "smart_5_raw_rolling_std_7d", "smart_5_raw_rolling_mean_30d"]

features = raw_features + history_features

train_end = pd.Timestamp("2026-02-09")
val_end = pd.Timestamp("2026-02-19")

X = df[features]
y = df["failure_within_30_days"]
train_mask = df["date"] <= train_end
val_mask = (df["date"] > train_end) & (df["date"] <= val_end)

X_train = X.loc[train_mask]
X_val = X.loc[val_mask]

y_train = y.loc[train_mask]
y_val = y.loc[val_mask]

pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("model", RandomForestClassifier(random_state=44))])

pipeline.fit(X_train, y_train)

failure_probabilities = pipeline.predict_proba(X_val)[:, 1]

predictions = (failure_probabilities >= 0.3).astype(int)

In [14]:
val_results = df.loc[val_mask].copy()

val_results["failure_probability"] = failure_probabilities
val_results["prediction"] = predictions

false_negatives = val_results[(val_results["failure_within_30_days"] == 1) & (val_results["prediction"] == 0)]
false_positives = val_results[(val_results["failure_within_30_days"] == 0) & (val_results["prediction"] == 1)]

print("False negatives:", len(false_negatives))
print("False positives:", len(false_positives))

print(false_negatives[["serial_number", "date", "failure_probability", "smart_5_raw", "smart_5_raw_change_7d", "smart_5_raw_change_30d"]].head())


print(false_positives[["serial_number", "date", "failure_probability", "smart_5_raw", "smart_5_raw_change_7d", "smart_5_raw_change_30d"]].head())

False negatives: 643
False positives: 164
      serial_number       date  failure_probability  smart_5_raw  \
69565  88P0A0JRF97G 2026-02-10                 0.01          0.0   
69589  8190A0E5FVKG 2026-02-10                 0.08         46.0   
69631  6280A0V8FVKG 2026-02-10                 0.00          0.0   
69726  88Q0A0BCF97G 2026-02-10                 0.03          0.0   
69919  6250A00XFVKG 2026-02-10                 0.01          0.0   

       smart_5_raw_change_7d  smart_5_raw_change_30d  
69565                    0.0                     NaN  
69589                   19.0                    31.0  
69631                    0.0                     0.0  
69726                    0.0                     0.0  
69919                    0.0                     0.0  
          serial_number       date  failure_probability  smart_5_raw  \
69732  1a1cf23811210010 2026-02-10             0.389424          NaN   
69814          1QHJJKAX 2026-02-10             0.350000          0.0   
699

In [15]:
from sklearn.inspection import permutation_importance

# Finds which features the model depends on when predicting the drive failures
importance_result = permutation_importance(pipeline, X_val, y_val, scoring="f1", n_repeats=5, random_state=44)

importance_df = pd.DataFrame({"Feature": features, "Importance": importance_result.importances_mean})
importance_df = importance_df.sort_values("Importance", ascending=False)


print(importance_df)

                         Feature  Importance
32  smart_5_raw_rolling_mean_30d    0.109606
29        smart_5_raw_change_30d    0.085660
30   smart_5_raw_rolling_mean_7d    0.043715
11                   smart_9_raw    0.038237
18                 smart_193_raw    0.032907
31    smart_5_raw_rolling_std_7d    0.026483
22                 smart_197_raw    0.019379
27         smart_5_raw_change_1d    0.018794
28         smart_5_raw_change_7d    0.017166
7                    smart_5_raw    0.009741
16                 smart_192_raw    0.008173
9                    smart_7_raw    0.006699
10            smart_9_normalized    0.006057
24                 smart_198_raw    0.005404
5                    smart_4_raw    0.005019
17          smart_193_normalized    0.003503
3                    smart_3_raw    0.003218
14                  smart_12_raw    0.002880
25          smart_199_normalized    0.002129
0             smart_1_normalized    0.001140
8             smart_7_normalized    0.001107
20        

### Feature Importance

The permutation importance showed that the most important features included the 30-day rolling average, 30-day change and the 7-day rolling average of SMART 5. These history features were more important than the current SMART 5 value by itself and this supports the result from earlier that adding information about how SMART values change over time improved the model.

In [16]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

test_mask = df["date"] > val_end
X_test = X.loc[test_mask]
y_test = y.loc[test_mask]

final_train_mask = df["date"] <= val_end
X_final_train = X.loc[final_train_mask]
y_final_train = y.loc[final_train_mask]


final_pipeline = Pipeline([("imputer", SimpleImputer(strategy= "median", add_indicator=True)),("model", RandomForestClassifier(random_state = 44))])


final_pipeline.fit(X_final_train, y_final_train)

test_failure_probabilities = final_pipeline.predict_proba(X_test)[:, 1]
test_predictions = (test_failure_probabilities >= 0.3).astype(int)
test_precision = precision_score(y_test, test_predictions)
test_recall = recall_score(y_test, test_predictions)
test_f1 = f1_score(y_test, test_predictions)
test_roc_auc = roc_auc_score(y_test, test_failure_probabilities)
test_pr_auc = average_precision_score(y_test, test_failure_probabilities)
tn, fp, fn, tp = confusion_matrix(y_test, test_predictions).ravel()

print("Precision:", test_precision)
print("Recall:", test_recall)
print("F1:", test_f1)
print("ROC-AUC:", test_roc_auc)
print("PR-AUC:", test_pr_auc)
print("False Positives:", fp)
print("False Negatives:", fn)

Precision: 0.9850085178875639
Recall: 0.7663353214049039
F1: 0.8620201267238167
ROC-AUC: 0.9785074456274095
PR-AUC: 0.9733774603695051
False Positives: 88
False Negatives: 1763


### Final Test Results

The final model was tested on unseen data using the 0.3 threshold. It got about 99% precision, 77% recall, and an 86% F1 score. The recall was a little lower than the validation results, but the model still performed well on data that was not used during training or model selection.

In [19]:
import joblib

joblib.dump(final_pipeline, "../models/pulse_model.joblib")
loaded_pipeline = joblib.load("../models/pulse_model.joblib")

loaded_probabilities = loaded_pipeline.predict_proba(X_test.head())[:, 1]
print(loaded_probabilities)

[0. 0. 1. 0. 0.]


In [20]:
needed_columns = list(final_pipeline.feature_names_in_) + ["serial_number", "date", "smart_5_raw", "smart_5_raw_change_1d", "smart_5_raw_change_7d", "smart_5_raw_change_30d"]
needed_columns = list(dict.fromkeys(needed_columns))
demo_df = df[needed_columns]
demo_df.to_csv("../data/processed/demo_data.csv", index=False)

print(demo_df.shape)

(102123, 35)
